In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta con los CSV generados por `transform`
CSV_DIR = Path("../data/csv")

REV = "estimatedRevenue"
# Métricas que se pueden sumar entre días y vídeos (cpm y playbackBasedCpm NO)
ADDITIVE = ["estimatedRevenue", "estimatedAdRevenue", "grossRevenue",
            "adImpressions", "monetizedPlaybacks"]
# Umbral de vistas para calcular el RPM mediano por vídeo sin ruido de vídeos minúsculos
MIN_VIEWS_RPM = 1000

daily = pd.read_csv(CSV_DIR / "analytics_channel_revenue_daily.csv", parse_dates=["day"])
chan = pd.read_csv(CSV_DIR / "analytics_channel_daily.csv", parse_dates=["day"])
vid_rev = pd.read_csv(CSV_DIR / "analytics_video_revenue_totals.csv")
perf = pd.read_csv(CSV_DIR / "video_performance.csv")
perf["published_at"] = pd.to_datetime(perf["published_at"], utc=True)

# --- 8.1 Estructura y calidad -----------------------------------------------
print("== 8.1 Estructura ==")
print(f"Serie diaria: {len(daily)} filas | {daily['day'].min().date()} -> {daily['day'].max().date()}")
print(f"Totales por vídeo: {len(vid_rev)} filas | vídeos distintos: {vid_rev['video_id'].nunique()}")
print("\nNulos en la serie diaria:")
print(daily[ADDITIVE + ["cpm", "playbackBasedCpm"]].isna().sum().to_string())
print(f"\nDías con {REV} negativo: {int((daily[REV] < 0).sum())}")

# --- 8.2 Totales del canal y arranque de la monetización ---------------------
print("\n== 8.2 Totales del canal (suma de la serie diaria) ==")
print(daily[ADDITIVE].sum().round(2).to_string())

con_ingresos = daily[daily[REV] > 0]
print(f"\nDías con {REV} > 0: {len(con_ingresos)} de {len(daily)}")
if not con_ingresos.empty:
    print(f"Primer día con ingresos: {con_ingresos['day'].min().date()}")
    print(f"Último día con ingresos: {con_ingresos['day'].max().date()}")

anual = daily.merge(chan[["day", "views"]], on="day", how="left")
anual["anio"] = anual["day"].dt.year
tabla_anual = anual.groupby("anio").agg(
    ingresos=(REV, "sum"),
    dias_con_ingresos=(REV, lambda s: int((s > 0).sum())),
    vistas=("views", "sum"),
)
tabla_anual["rpm"] = tabla_anual["ingresos"] / tabla_anual["vistas"].where(tabla_anual["vistas"] > 0) * 1000
print("\nIngresos, vistas y RPM del canal por año:")
print(tabla_anual.round(2).to_string())

# --- 8.3 Últimos días (posible retraso o cifras provisionales) ---------------
print("\n== 8.3 Últimos 10 días de la serie ==")
print(daily.sort_values("day").tail(10)[["day", REV, "estimatedAdRevenue", "monetizedPlaybacks"]]
      .round(2).to_string(index=False))

# --- 8.4 Canal vs suma de vídeos ----------------------------------------------
total_canal = daily[REV].sum()
total_videos = vid_rev[REV].sum()
print("\n== 8.4 Canal vs suma de vídeos ==")
print(f"Canal (serie diaria): {total_canal:,.2f}")
print(f"Suma de vídeos:       {total_videos:,.2f}")
print(f"Brecha: {total_canal - total_videos:,.2f} ({(total_canal - total_videos) / total_canal:.2%} del canal)")
print(f"Vídeos con {REV} > 0: {int((vid_rev[REV] > 0).sum())} de {len(vid_rev)}")

por_estado = perf[perf["views"].notna()].groupby("is_public")[REV].agg(["count", "sum"])
print("\nIngresos por vídeos públicos (True) / no públicos (False):")
print(por_estado.round(2).to_string())

# --- 8.5 Concentración de ingresos --------------------------------------------
ordenados = vid_rev[vid_rev[REV] > 0].sort_values(REV, ascending=False).reset_index(drop=True)
acum = ordenados[REV].cumsum() / ordenados[REV].sum()
print("\n== 8.5 Concentración (vídeos con ingresos) ==")
print(f"Vídeo nº1: {ordenados.loc[0, REV] / ordenados[REV].sum():.1%} de los ingresos")
print(f"Top 10:    {acum.iloc[min(9, len(acum) - 1)]:.1%}")
print(f"Vídeos necesarios para el 50%: {int((acum < 0.5).sum()) + 1}")
print(f"Vídeos necesarios para el 80%: {int((acum < 0.8).sum()) + 1}")

# --- 8.6 Ingresos y RPM por formato (públicos con Analytics) ------------------
pub = perf[perf["is_public"] & perf["views"].notna()].copy()
pub["rpm"] = pub[REV] / pub["views"].where(pub["views"] > 0) * 1000

por_formato = pub.groupby("format").agg(
    videos=("video_id", "count"),
    vistas=("views", "sum"),
    ingresos=(REV, "sum"),
)
por_formato["cuota_vistas_%"] = por_formato["vistas"] / por_formato["vistas"].sum() * 100
por_formato["cuota_ingresos_%"] = por_formato["ingresos"] / por_formato["ingresos"].sum() * 100
por_formato["rpm_agregado"] = por_formato["ingresos"] / por_formato["vistas"] * 1000
por_formato["rpm_mediano_por_video"] = (
    pub[pub["views"] >= MIN_VIEWS_RPM].groupby("format")["rpm"].median()
)
print(f"\n== 8.6 Por formato (públicos; RPM mediano con >= {MIN_VIEWS_RPM} vistas) ==")
print(por_formato.round(2).to_string())

# --- 8.7 Cobertura de monetización por año de publicación ---------------------
pub["anio_pub"] = pub["published_at"].dt.year
cobertura = pub.groupby("anio_pub").agg(
    videos=("video_id", "count"),
    con_ingresos=(REV, lambda s: int((s > 0).sum())),
)
cobertura["con_ingresos_%"] = cobertura["con_ingresos"] / cobertura["videos"] * 100
print("\n== 8.7 Vídeos públicos con ingresos > 0, por año de publicación ==")
print(cobertura.round(1).to_string())

# --- 8.8 Top 10 vídeos por ingresos -------------------------------------------
cols = ["video_id", "title", "format", "published_at", "views", REV, "rpm"]
print("\n== 8.8 Top 10 por ingresos (públicos) ==")
top = pub.sort_values(REV, ascending=False).head(10)[cols].copy()
top["published_at"] = top["published_at"].dt.date
top[[REV, "rpm"]] = top[[REV, "rpm"]].round(2)
print(top.to_string(index=False))

== 8.1 Estructura ==
Serie diaria: 4211 filas | 2015-04-11 -> 2026-10-20
Totales por vídeo: 371 filas | vídeos distintos: 371

Nulos en la serie diaria:
estimatedRevenue      0
estimatedAdRevenue    0
grossRevenue          0
adImpressions         0
monetizedPlaybacks    0
cpm                   0
playbackBasedCpm      0

Días con estimatedRevenue negativo: 0

== 8.2 Totales del canal (suma de la serie diaria) ==
estimatedRevenue         5074.82
estimatedAdRevenue       4693.72
grossRevenue             8534.11
adImpressions         2308521.00
monetizedPlaybacks    1310266.00

Días con estimatedRevenue > 0: 731 de 4211
Primer día con ingresos: 2017-05-24
Último día con ingresos: 2026-10-20

Ingresos, vistas y RPM del canal por año:
      ingresos  dias_con_ingresos   vistas   rpm
anio                                            
2015      0.00                  0     2440  0.00
2016      0.00                  0     1945  0.00
2017     10.00                192    62356  0.16
2018      0.30  

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5664\1911731565.py:54: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2).to_string(index=False))
